#### Imports 

In [22]:
import os
import warnings 
import random 
from zipfile import ZipFile 
from pathlib import Path 
import copy

import pandas as pd 
import sklearn 
import numpy as np

import matplotlib.pyplot as plt 
from tqdm.auto import tqdm 
from torchinfo import summary

import torch
from torch import nn 
from torch .utils.data import DataLoader, Dataset 
import torch.nn.functional as F
from torchvision import transforms 
from torchvision.transforms import v2
from sklearn.model_selection import StratifiedKFold 

import timm

warnings.filterwarnings("ignore")

In [23]:
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

In [24]:
device = torch.device("mps"
if torch.backends.mps.is_available()
else "cuda"
if torch.cuda.is_available()
else "cpu")

print(f"Device: '{device}'")

Device: 'mps'


In [25]:
SEED = 42
N_SPLITS = 5
BATCH_SIZE = 64
EPOCHS = 25
LR = 1e-3

In [26]:
def seed_everything(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic=(True)
    os.environ["PYTHONHASHSEED"] = str(seed)

In [27]:
seed_everything(SEED)

#### Download data

In [28]:
data_path = Path("data")
file_path = data_path / "digit_recognizer"
file_path_zip = data_path / "digit-recognizer.zip"

with ZipFile(file_path_zip, "r") as rep_zip:
    print(f"Unzipping digit recognizer data in {file_path}...")
    rep_zip.extractall(file_path)

Unzipping digit recognizer data in data/digit_recognizer...


#### Tensor data

In [29]:
# train_df = pd.read_csv("/kaggle/input/competitions/digit-recognizer/train.csv")
# test_df = pd.read_csv("/kaggle/input/competitions/digit-recognizer/test.csv")

#### Transforms

In [30]:
train_transforms = v2.Compose([
    v2.RandomRotation(degrees=10),
    v2.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.92, 1.08)),
])

#### Dataset

In [31]:
class DigitDataset(Dataset):

    def __init__(self, X, y=None, transform=None):
        self.X = X
        self.y = y
        self.transform = transform 

    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        x = self.X[idx]
        if self.transform:
            x = self.transform(x)
        if self.y is not None:
            return x, self.y[idx]
        return x    

#### Custom model

In [32]:
class ModernDigitCNN(nn.Module):

    def __init__(self, in_features: int=1, num_classes: int=10):
        super().__init__()

        def conv_block(in_f, out_f):
            return nn.Sequential(
                nn.Conv2d(in_channels=in_f, out_channels=out_f, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_f),
                nn.SiLU(),
                nn.Conv2d(in_channels=out_f, out_channels=out_f, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_f),
                nn.SiLU(),
                nn.MaxPool2d((2, 2)),
                nn.Dropout(0.15)
            )
        
        self.features = nn.Sequential(
            conv_block(in_f=in_features, out_f=32),
            conv_block(in_f=32, out_f=64),
            conv_block(in_f=64, out_f=128)
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x):
        return self.classifier(self.features(x))

#### Timm model

In [33]:
class PretrainedTimmModel(nn.Module):

    def __init__(
            self, model_name="resnet18.a1_in1k", num_classes=10
    ):
        super().__init__()

        self.model = timm.create_model(
            model_name, pretrained=True, in_chans=1, num_classes=num_classes)

    def forward(self, x):
        return self.model(x)

#### Train step

In [34]:
def train_epoch(model: torch.nn.Module,
                dataloader: torch.utils.data.DataLoader,
                optimizer: torch.optim.Optimizer,
                loss_fn: torch.nn.Module,
                device: torch.device):
    
    model.train()
    total_loss, correct = 0.0, 0

    for X, y in dataloader:

        X, y = X.to(device), y.to(device)

        if y.ndim > 1:
            y = y.squeeze(-1)

        loggits = model(X)

        loss = loss_fn(loggits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() / len(y)
        preds = torch.argmax(loggits, dim=1)
        correct += (preds == y).sum().item()
        
    return total_loss / len(dataloader.dataset), correct / len(dataloader.dataset)

#### Valid step

In [35]:
def eval_epoch(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               device: torch.device):
    
    model.eval()
    total_loss, correct = 0.0, 0

    for X, y in dataloader:
        X, y = X.to(device), y.to(device)

        if y.ndim > 1:
            y = y.squeeze(-1)

        loggits = model(X)

        loss = loss_fn(loggits, y)
        total_loss += loss.item() / len(y)
        preds = torch.argmax(loggits, dim=1)
        correct += (preds == y).sum().item()
        
    return total_loss / len(dataloader.dataset), correct / len(dataloader.dataset)

#### Cross-validation train 

In [ ]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

def train_model(model_cls,
          X_train_all,
          y_train_all,
          batch_size=BATCH_SIZE,
          epochs=EPOCHS,
          lr=LR,
          n_splits=N_SPLITS):
    
    models = []

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(X_train_all, y_train_all)
    ):
        print(f"\n--- Fold {fold + 1}/{n_splits} ---")

        train_sub = DigitDataset(
            X_train_all[train_idx],
            y_train_all[train_idx],
            transform=train_transforms)
        
        val_sub = DigitDataset(X_train_all[val_idx],
                            y_train_all[val_idx],
                            transform=None)

        train_loader = DataLoader(
            train_sub,
            batch_size=batch_size,
            shuffle=True,
            num_workers=0)
        
        val_loader = DataLoader(
            val_sub,
            batch_size=batch_size,
            shuffle=False,
            num_workers=0
        )

        model = model_cls().to(device)

        optimizer = torch.optim.AdamW(
            params=model.parameters(),
            lr=lr,
            weight_decay=1e-2
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=epochs
        )
        loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=0.1)

        best_val_acc = 0.0
        best_model_weights = None 

        for epoch in range(epochs):
            train_loss, train_acc = train_epoch(
                model, train_loader, optimizer, loss_fn, device
            )
            val_loss, val_acc = eval_epoch(
                model, val_loader, loss_fn, device
            )
            scheduler.step()

            if val_acc > best_val_acc:
                best_val_acc = val_acc 
                best_model_weights = copy.deepcopy(model.state_dict())

            if (epoch+1) % 5 == 0 or epoch == epochs -1:
                print(
                f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {train_loss:.4f} | Acc:"
                f" {train_acc:.4f} | Val Loss: {val_loss:.4f} | Acc: {val_acc:.4f}"
        )
        print(
            f"--> Fold {fold + 1} Best Val Accuracy: {best_val_acc:.5f}"
        )

        model.load_state_dict(best_model_weights)
        models.append(model)

        model.eval()
        val_probs = []

        with torch.inference_mode():

            for X_val, _ in val_loader:
                X_val = X_val.to(device)

                loggits = model(X_val)
                probs = torch.softmax(loggits, dim=1).cpu().numpy()
                val_probs.append(probs)

    return models

# weights_path = Path("weights-pro+")
# weights_path.mkdir(parents=True, exist_ok=True)
# weights_file = weights_path / "weights-ModernDigitCNN.pth"

# torch.save([m.state_dict() for m in models], weights_file)
# print(f"Weights saved in {weights_file}")

#### TTA predict function 

In [37]:
def predict_tta(model, test_loader):

    model.eval()
    probs_list = []

    with torch.inference_mode():

        for X_batch in test_loader:

            X_orig = X_batch.to(device)

            p0 = F.softmax(model(X_orig), dim=1)

            X_rot1 = v2.functional.rotate(X_orig, angle=5).to(device)
            p1 = F.softmax(model(X_rot1), dim=1)
            
            X_rot2 = v2.functional.rotate(X_orig, angle=-5).to(device)
            p2 = F.softmax(model(X_rot2), dim=1)

            probs_list.append(((p0+p1+p2)/3).cpu().numpy())

    return np.vstack(probs_list)

#### Test data tensor

In [38]:
train_dir = file_path / "train.csv"
test_dir = file_path / "test.csv"

train_df = pd.read_csv(train_dir)
test_df = pd.read_csv(test_dir)

X_raw = train_df.drop(columns=["label"]).values 
y_raw = train_df["label"].values
X_test_raw = test_df.values 

X_train_all = (
    torch.tensor(X_raw, dtype=torch.float32).reshape(-1, 1, 28, 28) / 255.0
)
y_train_all = (
    torch.tensor(y_raw, dtype=torch.long)
)
X_test_all = (
    torch.tensor(X_test_raw, dtype=torch.float32).reshape(-1, 1, 28, 28) / 255.0
)

print(f"X Train tensor shape: {X_train_all.shape}")
print(f"Y Train shape: {y_train_all.shape}")
print(f"X Test shape: {X_test_all.shape}") 

X Train tensor shape: torch.Size([42000, 1, 28, 28])
Y Train shape: torch.Size([42000])
X Test shape: torch.Size([28000, 1, 28, 28])


#### Test data dataset and dataloader

In [39]:
test_dataset = DigitDataset(X_test_all, transform=None)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0)

#### TTA test predict ansamble

In [ ]:
print(f"\nFit custom CNN models")
print(f"=== Start {N_SPLITS}-Fold Cross-Validation ")
cnn_models = train_model(
    ModernDigitCNN, X_train_all, y_train_all, epochs=25, lr=1e-3
)

print(f"Fit Pretrained ConvNexT models")
print(f"=== Start {N_SPLITS}-Fold Cross-Validation ")
convnext_models = train_model(
    PretrainedTimmModel, X_train_all, y_train_all, epochs=15, lr=5e-4
)

final_preds = np.zeros((len(X_test_all), 10))

for model in cnn_models:
    final_preds += (predict_tta(model, test_loader) / 5.0) * 0.5 

for model in convnext_models:
    final_preds += (predict_tta(model, test_loader) / 5.0) * 0.5

final_preds = final_preds.argmax(axis=1)

Fit custom CNN models
=== Start 5-Fold Cross-Validation 

--- Fold 1/5 ---
Epoch 01/25 | Train Loss: 0.0002 | Acc: 0.9335 | Val Loss: 0.0001 | Acc: 0.9850
Epoch 02/25 | Train Loss: 0.0001 | Acc: 0.9836 | Val Loss: 0.0001 | Acc: 0.9900
Epoch 03/25 | Train Loss: 0.0001 | Acc: 0.9876 | Val Loss: 0.0001 | Acc: 0.9926
Epoch 04/25 | Train Loss: 0.0001 | Acc: 0.9892 | Val Loss: 0.0001 | Acc: 0.9927
Epoch 05/25 | Train Loss: 0.0001 | Acc: 0.9903 | Val Loss: 0.0001 | Acc: 0.9925
Epoch 06/25 | Train Loss: 0.0001 | Acc: 0.9914 | Val Loss: 0.0001 | Acc: 0.9940
Epoch 07/25 | Train Loss: 0.0001 | Acc: 0.9919 | Val Loss: 0.0001 | Acc: 0.9940
Epoch 08/25 | Train Loss: 0.0001 | Acc: 0.9932 | Val Loss: 0.0001 | Acc: 0.9939
Epoch 09/25 | Train Loss: 0.0001 | Acc: 0.9935 | Val Loss: 0.0001 | Acc: 0.9943
Epoch 10/25 | Train Loss: 0.0001 | Acc: 0.9937 | Val Loss: 0.0001 | Acc: 0.9940
Epoch 11/25 | Train Loss: 0.0001 | Acc: 0.9934 | Val Loss: 0.0001 | Acc: 0.9944
Epoch 12/25 | Train Loss: 0.0001 | Acc: 0.995

In [ ]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

def train_model(model_cls,
          X_train_all,
          y_train_all,
          batch_size=BATCH_SIZE,
          epochs=EPOCHS,
          lr=LR,
          n_splits=N_SPLITS):
    
    models = []

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(X_train_all, y_train_all)
    ):
        print(f"\n--- Fold {fold + 1}/{n_splits} ---")

        train_sub = DigitDataset(
            X_train_all[train_idx],
            y_train_all[train_idx],
            transform=train_transforms)
        
        val_sub = DigitDataset(X_train_all[val_idx],
                            y_train_all[val_idx],
                            transform=None)

        train_loader = DataLoader(
            train_sub,
            batch_size=batch_size,
            shuffle=True,
            num_workers=0)
        
        val_loader = DataLoader(
            val_sub,
            batch_size=batch_size,
            shuffle=False,
            num_workers=0
        )

        model = model_cls().to(device)

        optimizer = torch.optim.AdamW(
            params=model.parameters(),
            lr=lr,
            weight_decay=1e-2
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=epochs
        )
        loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=0.1)

        best_val_acc = 0.0
        best_model_weights = None 

        for epoch in range(epochs):
            train_loss, train_acc = train_epoch(
                model, train_loader, optimizer, loss_fn, device
            )
            val_loss, val_acc = eval_epoch(
                model, val_loader, loss_fn, device
            )
            scheduler.step()

            if val_acc > best_val_acc:
                best_val_acc = val_acc 
                best_model_weights = copy.deepcopy(model.state_dict())

            # if (epoch+1) % 5 == 0 or epoch == epochs -1:
            print(
            f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {train_loss:.4f} | Acc:"
            f" {train_acc:.4f} | Val Loss: {val_loss:.4f} | Acc: {val_acc:.4f}"
        )
        print(
            f"--> Fold {fold + 1} Best Val Accuracy: {best_val_acc:.5f}"
        )

        model.load_state_dict(best_model_weights)
        models.append(model)

        model.eval()
        val_probs = []

        with torch.inference_mode():

            for X_val, _ in val_loader:
                X_val = X_val.to(device)

                loggits = model(X_val)
                probs = torch.softmax(loggits, dim=1).cpu().numpy()
                val_probs.append(probs)

    return models

# weights_path = Path("weights-pro+")
# weights_path.mkdir(parents=True, exist_ok=True)
# weights_file = weights_path / "weights-ModernDigitCNN.pth"

# torch.save([m.state_dict() for m in models], weights_file)
# print(f"Weights saved in {weights_file}")

#### Prediction 

In [ ]:
print(f"Generation ansamble-result predictions...")

submission = pd.DataFrame({"ImageId": range(1, len(final_preds) + 1),
                           "Label": final_preds})

submission.to_csv("submission.csv", index=False)
submission.head(10)

Generation ansamble-result predictions...


,ImageId,Label
0,1,2
1,2,0
2,3,9
3,4,0
4,5,3
5,6,7
6,7,0
7,8,3
8,9,0
9,10,3
